# AI Based Career Prediction and Personalized Skill Gap Analysis System Using Machine Learning and NLP

This Google Colab notebook contains:
- Dataset
- Dataset inspection
- Career distribution graph
- TF-IDF NLP
- Decision Tree classifier
- Accuracy
- Classification report
- Confusion matrix
- Decision Tree visualization
- Feature importance
- New student career prediction
- Personalized skill-gap analysis
- Learning roadmap


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

print("All libraries imported successfully!")

In [ ]:
data = {
    "skills": [
        "python machine learning artificial intelligence deep learning pandas numpy statistics",
        "python tensorflow keras machine learning neural networks pandas numpy",
        "python sklearn machine learning deep learning tensorflow numpy",
        "python artificial intelligence machine learning statistics pandas",
        "python pandas numpy sql excel statistics data analysis",
        "sql excel power bi tableau data analysis statistics",
        "python sql excel power bi data visualization",
        "excel sql pandas python data visualization power bi",
        "html css javascript react frontend web development",
        "html css javascript bootstrap frontend development",
        "javascript react html css responsive web design",
        "html css javascript react web development",
        "aws cloud computing linux networking docker",
        "aws azure cloud computing docker kubernetes",
        "cloud computing linux networking aws server",
        "azure cloud docker kubernetes networking",
        "python django flask sql backend development api",
        "java spring boot sql backend development",
        "python flask sql api database backend",
        "java spring sql database backend development",
        "python java data structures algorithms programming",
        "c programming data structures algorithms problem solving",
        "java algorithms data structures problem solving",
        "python programming algorithms data structures"
    ],
    "career": [
        "AI/ML Engineer","AI/ML Engineer","AI/ML Engineer","AI/ML Engineer",
        "Data Analyst","Data Analyst","Data Analyst","Data Analyst",
        "Frontend Developer","Frontend Developer","Frontend Developer","Frontend Developer",
        "Cloud Engineer","Cloud Engineer","Cloud Engineer","Cloud Engineer",
        "Backend Developer","Backend Developer","Backend Developer","Backend Developer",
        "Software Developer","Software Developer","Software Developer","Software Developer"
    ]
}

df = pd.DataFrame(data)

print("Dataset created successfully!")
print("Dataset Shape:", df.shape)
df.head(10)

In [ ]:
pd.set_option("display.max_colwidth", None)
df

In [ ]:
print("Dataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
df = df.dropna()
print("Missing values removed successfully!")
print("Final Dataset Shape:", df.shape)

In [ ]:
career_counts = df["career"].value_counts()
print(career_counts)

In [ ]:
plt.figure(figsize=(10, 6))
career_counts.plot(kind="bar")
plt.xlabel("Career Role")
plt.ylabel("Number of Students")
plt.title("Career Role Distribution")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
X = df["skills"]
y = df["career"]

print("Input Features:")
print(X.head())

print("\nTarget Career:")
print(y.head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

In [ ]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF Vectorization Completed!")
print("Training Vector Shape:", X_train_tfidf.shape)
print("Testing Vector Shape:", X_test_tfidf.shape)

In [ ]:
decision_tree = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=6,
    random_state=42
)

decision_tree.fit(X_train_tfidf, y_train)

print("Decision Tree Model Training Completed!")

In [ ]:
y_pred = decision_tree.predict(X_test_tfidf)

print("Actual Careers:")
print(y_test.values)

print("\nPredicted Careers:")
print(y_pred)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print("Decision Tree Accuracy:", round(accuracy * 100, 2), "%")

In [ ]:
print("Classification Report:\n")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
cm = confusion_matrix(
    y_test, y_pred, labels=decision_tree.classes_
)

print("Confusion Matrix:")
print(cm)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=decision_tree.classes_
)

disp.plot(ax=ax, xticks_rotation=45)
plt.title("Career Prediction - Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(25, 15))

plot_tree(
    decision_tree,
    feature_names=vectorizer.get_feature_names_out(),
    class_names=decision_tree.classes_,
    filled=True,
    rounded=True,
    fontsize=8
)

plt.title("Decision Tree for Career Prediction")
plt.show()

In [ ]:
feature_names = vectorizer.get_feature_names_out()
importance = decision_tree.feature_importances_

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
}).sort_values(by="Importance", ascending=False)

print("Top Important Features:")
print(feature_importance.head(15))

In [ ]:
top_features = feature_importance.head(10)

plt.figure(figsize=(10, 6))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top Features Used for Career Prediction")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
career_skills = {
    "AI/ML Engineer": {
        "python","machine learning","deep learning","tensorflow",
        "pytorch","pandas","numpy","statistics"
    },
    "Data Analyst": {
        "python","sql","excel","pandas","numpy","statistics",
        "power bi","data visualization"
    },
    "Frontend Developer": {
        "html","css","javascript","react","bootstrap","responsive design"
    },
    "Cloud Engineer": {
        "aws","azure","cloud computing","linux","docker",
        "kubernetes","networking"
    },
    "Backend Developer": {
        "python","java","sql","django","flask","spring","api","database"
    },
    "Software Developer": {
        "python","java","c programming","data structures",
        "algorithms","problem solving"
    }
}

print("Career skill database created successfully!")

In [ ]:
def predict_career(student_skills):
    student_vector = vectorizer.transform([student_skills])
    return decision_tree.predict(student_vector)[0]

def skill_gap_analysis(student_skills, predicted_career):
    user_skills = set(
        skill.strip().lower()
        for skill in student_skills.split(",")
    )
    required_skills = career_skills[predicted_career]
    matched_skills = user_skills.intersection(required_skills)
    missing_skills = required_skills.difference(user_skills)
    match_percentage = (
        len(matched_skills) / len(required_skills)
    ) * 100
    return matched_skills, missing_skills, match_percentage

In [ ]:
def create_roadmap(missing_skills):
    roadmap = []
    for skill in sorted(missing_skills):
        roadmap.append(f"Learn {skill}")
    return roadmap

In [ ]:
# Change this input to test another student.
new_student = "python, pandas, numpy, sql, statistics"

predicted_career = predict_career(new_student)
matched_skills, missing_skills, match_percentage = skill_gap_analysis(
    new_student, predicted_career
)
roadmap = create_roadmap(missing_skills)

print("=" * 70)
print("AI BASED CAREER PREDICTION AND PERSONALIZED SKILL GAP ANALYSIS")
print("=" * 70)

print("\nStudent Skills:")
print(new_student)

print("\nPredicted Career:")
print(predicted_career)

print("\nSkill Match:")
print(round(match_percentage, 2), "%")

print("\nMatched Skills:")
for skill in sorted(matched_skills):
    print("✓", skill)

print("\nMissing Skills:")
for skill in sorted(missing_skills):
    print("✗", skill)

print("\nPersonalized Learning Roadmap:")
for i, item in enumerate(roadmap, 1):
    print(f"{i}. {item}")

print("=" * 70)

In [ ]:
# Example: test another new student
new_student = "html, css, javascript, react"

predicted_career = predict_career(new_student)
matched_skills, missing_skills, match_percentage = skill_gap_analysis(
    new_student, predicted_career
)

print("New Student Career Prediction:", predicted_career)
print("Skill Match:", round(match_percentage, 2), "%")
print("\nMissing Skills:")
for skill in sorted(missing_skills):
    print("-", skill)